# Retrieving pid4cat metadata from the hdl.net API with Python

Details for the endpoints provided by the API can be found in the [auto-generated documentation](https://api.nfdi4cat.org/testpid/v2/index.html).

In [1]:
import logging

import httpx

from pid4cat_model.handle_api import (
    HandleAPIRecord,
    HandleConfig,
    HandleNetAPI,
    pid4cat_record_factory,
)

In [2]:
logger = logging.getLogger(__name__)
# Set logging level (DEBUG, INFO, WARNING, ERROR, CRITICAL)
logging.basicConfig(level=logging.INFO)

Setup a httpx-client (no authorisation required) and create the configuration for the namespace of the handle.

In [3]:
hdl_api_client = httpx.Client()
config = HandleConfig(ns_suffix="test-d4y")
config

HandleConfig(api_url='https://hdl.handle.net/api/handles/', prefix='21.T11978', ns_suffix='test-d4y', base_url='https://hdl.handle.net/api/handles/21.T11978/4cat/test-d4y/')

In addition to the config, we need to specify the id-suffix of the pid4cat handle:

In [4]:
id_suffix = "kbka-bem9"

Now we are ready to request the metadata for this handle identifier (21.T11978/4cat/test-d4y/kbka-bem9). It is a permanent demo handle on the testbed prefix 21.T11978, kept for this notebook.

In [5]:
api = HandleNetAPI(config, hdl_api_client)
metadata = api.get_metadata_for_id(id_suffix)

INFO:httpx:HTTP Request: GET https://hdl.handle.net/api/handles/21.T11978/4cat/test-d4y/kbka-bem9 "HTTP/1.1 200 OK"


For ease of use (and validation) we create a Pid4CatRecord instance via a factory function.

In [6]:
pid_rec = pid4cat_record_factory(metadata)

All metadata elements are easily accessible as attributes of the instance.

In [7]:
available = ", ".join(pid_rec.model_dump().keys())
print(f"All attributes  : {available}")
print(f"Landing page URL: {pid_rec.landing_page_url}")
print(f"Status          : {pid_rec.status}")
print(f"Curation contact: {pid_rec.curation_contact}")

All attributes  : landing_page_url, status, schema_version, metadata_license, curation_contact, resource_info, related_identifiers, change_log
Landing page URL: https://github.com/nfdi4cat/pid4cat-model/releases/tag/v0.4.4.1
Status          : REGISTERED
Curation contact: admin@pid4cat.org


We may also instantiate other classes from the generated pydantic data model, for example an instance of HandleAPIRecord, which closely matches the data model of the handle system.

However, for applications the above Pid4CatRecord instances are much more useful.

In [8]:
print("\nTrying to create a HandleAPIRecord object", end="... ")
hdl_rec = HandleAPIRecord.model_validate(metadata)
print("success!")
print(hdl_rec.model_dump_json(indent=2))


Trying to create a HandleAPIRecord object... success!
{
  "responseCode": 1,
  "handle": "21.T11978/4cat/test-d4y/kbka-bem9",
  "values": [
    {
      "timestamp": "2026-11-09T23:32:54Z",
      "ttl": 86400,
      "type": "URL",
      "index": 1,
      "data": {
        "format": "string",
        "value": "https://github.com/nfdi4cat/pid4cat-model/releases/tag/v0.4.4.1"
      }
    },
    {
      "timestamp": "2026-11-09T23:32:54Z",
      "ttl": 86400,
      "type": "EMAIL",
      "index": 10,
      "data": {
        "format": "string",
        "value": "admin@pid4cat.org"
      }
    },
    {
      "timestamp": "2026-11-09T23:32:54Z",
      "ttl": 86400,
      "type": "STATUS",
      "index": 11,
      "data": {
        "format": "string",
        "value": "REGISTERED"
      }
    },
    {
      "timestamp": "2026-11-09T23:32:54Z",
      "ttl": 86400,
      "type": "SCHEMA_VER",
      "index": 12,
      "data": {
        "format": "string",
        "value": "v0.4.4"
      }
    },
